# Section 05: Trust and Security with Google Cloud

**Companion notebook for**: `05-trust-security.html`

## Learning Objectives
- Understand cybersecurity threat types and Google Cloud defenses
- Work with IAM roles, policies, and service accounts
- Explore encryption options (Google-managed, CMEK, CSEK)
- Understand network security (VPC, firewall, Cloud Armor)
- Review compliance frameworks and governance tools

## Prerequisites
- A Google Cloud project (optional — mock outputs provided)
- Python 3.10+

---
## Setup & Dependencies

In [ ]:
%pip install -q google-cloud-iam google-cloud-kms google-cloud-resource-manager

In [ ]:
import os
import json

PROJECT_ID = os.environ.get("GCP_PROJECT_ID", "your-project-id")
print(f"Project ID: {PROJECT_ID}")

if PROJECT_ID == "your-project-id":
    print("WARNING: Mock outputs will be used.")

---
## Section 1: Cybersecurity Threat Landscape

Understanding common threats helps you choose the right defenses.

In [ ]:
# Threat types and Google Cloud defenses
threats = [
    {
        "threat": "DDoS Attack",
        "description": "Overwhelm service with massive traffic",
        "defense": "Cloud Armor + Global Load Balancer + Google Front End",
        "exam_tip": "Cloud Armor for HTTP/S, Google Front End for all traffic",
    },
    {
        "threat": "Phishing",
        "description": "Trick users into revealing credentials",
        "defense": "2FA, Titan Security Keys, BeyondCorp, Advanced Protection",
        "exam_tip": "Titan Security Keys = phishing-resistant hardware 2FA",
    },
    {
        "threat": "Data Breach",
        "description": "Unauthorized access to sensitive data",
        "defense": "IAM least privilege, VPC Service Controls, DLP, encryption",
        "exam_tip": "VPC Service Controls prevents data exfiltration via APIs",
    },
    {
        "threat": "Ransomware",
        "description": "Encrypt victim's data and demand ransom",
        "defense": "Immutable backups, SCC threat detection, Cloud Storage retention",
        "exam_tip": "Bucket Lock makes retention permanent (immutable)",
    },
    {
        "threat": "Insider Threat",
        "description": "Malicious or negligent employee actions",
        "defense": "Least privilege IAM, Audit Logs, Access Transparency",
        "exam_tip": "Access Transparency logs when Google staff access your data",
    },
    {
        "threat": "Supply Chain Attack",
        "description": "Compromise software dependencies",
        "defense": "Binary Authorization, Artifact Registry scanning",
        "exam_tip": "Binary Authorization = only deploy signed container images",
    },
]

print("Cybersecurity Threats and Google Cloud Defenses")
print("=" * 70)
for t in threats:
    print(f"\nThreat: {t['threat']}")
    print(f"  What: {t['description']}")
    print(f"  Defense: {t['defense']}")
    print(f"  Exam tip: {t['exam_tip']}")

---
## Section 2: IAM — Identity and Access Management

IAM controls WHO can do WHAT on WHICH resource.

In [ ]:
# IAM concepts and role types
print("IAM Components")
print("=" * 60)
components = [
    ("Member (Principal)", "WHO", "Google Account, Service Account, Group, Domain"),
    ("Role", "WHAT (permissions)", "Collection of API permissions"),
    ("Resource", "WHERE", "Project, bucket, VM, dataset, etc."),
    ("Policy", "BINDING", "Connects member + role on a resource"),
]
for name, what, desc in components:
    print(f"  {name:<22} ({what:<5}) — {desc}")

print("\n\nIAM Role Types")
print("=" * 60)
role_types = [
    {
        "type": "Basic Roles",
        "examples": "roles/viewer, roles/editor, roles/owner",
        "scope": "Broad — all services in project",
        "recommendation": "AVOID in production (too permissive)",
    },
    {
        "type": "Predefined Roles",
        "examples": "roles/bigquery.dataEditor, roles/storage.objectViewer",
        "scope": "Fine-grained, per-service",
        "recommendation": "RECOMMENDED for most use cases",
    },
    {
        "type": "Custom Roles",
        "examples": "roles/mycompany.customRole",
        "scope": "User-defined permissions",
        "recommendation": "When predefined roles are too broad",
    },
]
for r in role_types:
    print(f"\n  {r['type']}")
    print(f"    Examples: {r['examples']}")
    print(f"    Scope:    {r['scope']}")
    print(f"    Use:      {r['recommendation']}")

In [ ]:
# IAM gcloud commands
iam_commands = [
    ("List IAM policy", "gcloud projects get-iam-policy my-project"),
    ("Add role to user", "gcloud projects add-iam-policy-binding my-project --member='user:alice@example.com' --role='roles/viewer'"),
    ("Remove role", "gcloud projects remove-iam-policy-binding my-project --member='user:alice@example.com' --role='roles/viewer'"),
    ("Create service account", "gcloud iam service-accounts create my-sa --display-name='My Service Account'"),
    ("Grant role to SA", "gcloud projects add-iam-policy-binding my-project --member='serviceAccount:my-sa@my-project.iam.gserviceaccount.com' --role='roles/storage.objectViewer'"),
    ("List service accounts", "gcloud iam service-accounts list"),
    ("List available roles", "gcloud iam roles list --filter='name:bigquery'"),
]

print("Common IAM gcloud Commands")
print("=" * 70)
for desc, cmd in iam_commands:
    print(f"\n  # {desc}")
    print(f"  $ {cmd}")

---
## Section 3: Encryption Options

Google Cloud encrypts all data at rest by default. You can choose
who manages the encryption keys.

In [ ]:
# Encryption key management options
encryption_options = [
    {
        "option": "Google-managed (default)",
        "who_manages": "Google",
        "customer_action": "None — automatic and transparent",
        "use_case": "Most workloads, no regulatory key requirements",
        "cost": "Free (included)",
    },
    {
        "option": "CMEK (Customer-Managed Encryption Keys)",
        "who_manages": "Customer via Cloud KMS",
        "customer_action": "Create key in Cloud KMS, reference in service config",
        "use_case": "Regulatory compliance, key lifecycle control, audit trails",
        "cost": "Cloud KMS key usage fees",
    },
    {
        "option": "CSEK (Customer-Supplied Encryption Keys)",
        "who_manages": "Customer provides keys directly",
        "customer_action": "Supply key with every API request, manage key storage",
        "use_case": "Maximum control, key never stored by Google",
        "cost": "No KMS cost, but high operational burden",
    },
    {
        "option": "EKM (External Key Manager)",
        "who_manages": "Customer's external KMS (e.g., Thales, Fortanix)",
        "customer_action": "Configure external KMS integration via Cloud EKM",
        "use_case": "Data sovereignty, keys never enter Google infrastructure",
        "cost": "Cloud EKM + external KMS fees",
    },
]

print("Encryption Key Management Options")
print("=" * 70)
for opt in encryption_options:
    print(f"\n{opt['option']}")
    print(f"  Managed by:  {opt['who_manages']}")
    print(f"  Action:      {opt['customer_action']}")
    print(f"  Use case:    {opt['use_case']}")
    print(f"  Cost:        {opt['cost']}")

print("\nKey facts:")
print("  - All data at rest: AES-256 encryption (always, by default)")
print("  - All data in transit between Google DCs: ALTS encryption")
print("  - All external traffic: TLS/HTTPS")
print("  - Encryption CANNOT be disabled on Google Cloud")

---
## Section 4: Network Security

Google Cloud provides multiple layers of network protection.

In [ ]:
# Network security services
network_security = [
    ("VPC", "Virtual Private Cloud — isolated network for resources"),
    ("Firewall Rules", "Allow/deny traffic at VM level (ingress/egress)"),
    ("Cloud Armor", "DDoS protection + WAF for HTTP(S) load balancers"),
    ("VPC Service Controls", "API-level security perimeter, prevents data exfiltration"),
    ("Cloud NAT", "Outbound internet access for VMs without public IPs"),
    ("Cloud IDS", "Network intrusion detection (Palo Alto engine)"),
    ("Private Google Access", "Access Google APIs without public IP"),
    ("Shared VPC", "Centrally manage network across multiple projects"),
]

print("Network Security Services")
print("=" * 70)
for name, desc in network_security:
    print(f"  {name:<25} {desc}")

print("\nCloud Armor Features:")
armor_features = [
    "Preconfigured WAF rules (OWASP Top 10)",
    "IP allowlist / denylist (CIDR ranges)",
    "Geo-based access control (block by country)",
    "Rate limiting (throttle abusive clients)",
    "Adaptive Protection (ML-based anomaly detection)",
    "Bot management",
]
for f in armor_features:
    print(f"  - {f}")

---
## Section 5: Security Operations Services

Google Cloud provides integrated security monitoring, threat detection,
and compliance tools.

In [ ]:
# Security operations service map
security_services = [
    ("Security Command Center", "Posture management", "Asset inventory, vulnerability scanning, threat detection"),
    ("Chronicle SIEM", "Log analysis", "Petabyte-scale security log analysis, 12-month retention"),
    ("Cloud DLP", "Data protection", "Discover, classify, de-identify PII and sensitive data"),
    ("Cloud Audit Logs", "Activity logging", "Who did what, where, when (Admin + Data Access)"),
    ("Access Transparency", "Google access logs", "Logs when Google staff access your data"),
    ("Binary Authorization", "Deploy-time security", "Only deploy signed/trusted container images"),
    ("Secret Manager", "Secrets vault", "Store API keys, passwords, certs with versioning"),
    ("Cloud IDS", "Intrusion detection", "Network threat detection"),
    ("Certificate Manager", "SSL/TLS management", "Provision and manage SSL certificates"),
    ("BeyondCorp Enterprise", "Zero trust", "Context-aware access without VPN"),
]

print("Security Services Quick Reference")
print("=" * 85)
print(f"{'Service':<28} {'Category':<20} {'Key Capability'}")
print("-" * 85)
for name, cat, desc in security_services:
    print(f"{name:<28} {cat:<20} {desc}")

print("\nExam decision guide:")
print("  'Protect against DDoS' -> Cloud Armor")
print("  'Find PII in data' -> Cloud DLP")
print("  'Security posture dashboard' -> Security Command Center")
print("  'Store passwords securely' -> Secret Manager")
print("  'Zero-trust access' -> BeyondCorp Enterprise")

---
## Section 6: Compliance Frameworks

Google Cloud maintains certifications for major compliance frameworks.

In [ ]:
# Compliance frameworks
frameworks = [
    ("SOC 1/2/3", "Global", "Controls over financial reporting and security"),
    ("ISO 27001", "Global", "Information security management systems"),
    ("ISO 27017", "Global", "Cloud-specific security controls"),
    ("ISO 27018", "Global", "PII protection in public clouds"),
    ("GDPR", "EU", "Data protection for EU residents"),
    ("HIPAA", "US", "Protected health information (healthcare)"),
    ("PCI DSS", "Global", "Payment card data security"),
    ("FedRAMP", "US Gov", "Cloud security for federal agencies"),
    ("CSA STAR", "Global", "Cloud Security Alliance audit"),
]

print("Compliance Frameworks Supported by Google Cloud")
print("=" * 70)
print(f"{'Framework':<15} {'Region':<12} {'Covers'}")
print("-" * 70)
for name, region, desc in frameworks:
    print(f"{name:<15} {region:<12} {desc}")

print("\nKey governance tools:")
print("  Organization Policy Service — enforce constraints across all projects")
print("  Assured Workloads — compliance controls for regulated industries")
print("  Data residency controls — restrict data to specific regions")
print("  Audit Logs — who did what, when (Admin Activity always on, free)")

print("\nExam tip: Compliance is SHARED responsibility.")
print("  Google provides compliant infrastructure.")
print("  YOU must configure and use it correctly.")

---
## Summary

In this notebook we covered:

1. **Threats** — DDoS, phishing, ransomware, data breach, insider threat, supply chain attacks.
2. **IAM** — Members, roles (basic/predefined/custom), policies, service accounts, least privilege.
3. **Encryption** — Google-managed (default), CMEK, CSEK, EKM. All data encrypted at rest (AES-256).
4. **Network Security** — VPC, firewall rules, Cloud Armor (DDoS/WAF), VPC Service Controls.
5. **Security Services** — SCC, Chronicle, DLP, Audit Logs, Binary Authorization, Secret Manager.
6. **Compliance** — SOC, ISO 27001, GDPR, HIPAA, PCI DSS, FedRAMP. Shared responsibility.

**Next notebook**: Section 06 covers operations — cost management, billing, SRE, DevOps, monitoring, and sustainability.